In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
from typing import Tuple, List, Optional
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
import os
from langchain_community.graphs import Neo4jGraph
from langchain.document_loaders import WikipediaLoader
from langchain.text_splitter import TokenTextSplitter
from langchain_experimental.graph_transformers import LLMGraphTransformer
from neo4j import GraphDatabase
from yfiles_jupyter_graphs import GraphWidget
from langchain_community.vectorstores import Neo4jVector
from langchain_community.vectorstores.neo4j_vector import remove_lucene_chars
from langchain_ollama import OllamaLLM
from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
from typing import Tuple, List, Optional
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
import os
from langchain_community.graphs import Neo4jGraph
from langchain.document_loaders import WikipediaLoader
from langchain.text_splitter import TokenTextSplitter

from langchain_experimental.graph_transformers import LLMGraphTransformer
from neo4j import GraphDatabase
from yfiles_jupyter_graphs import GraphWidget
from langchain_community.vectorstores import Neo4jVector

from langchain_community.vectorstores.neo4j_vector import remove_lucene_chars
from langchain_core.runnables import ConfigurableField, RunnableParallel, RunnablePassthrough


In [4]:
os.environ["NEO4J_URI"] = "bolt://3.237.99.102:7687"
os.environ["NEO4J_USERNAME"] = "neo4j"
os.environ["NEO4J_PASSWORD"] = "slash-ceilings-misfit"

graph = Neo4jGraph()

In [5]:
real = WikipediaLoader(query = 'Empty Nest Syndrome').load()

text_split = TokenTextSplitter(chunk_size = 512, chunk_overlap = 24)
docs = text_split.split_documents(real[:2])

c:\Users\harsh\AppData\Local\Programs\Python\Python39\lib\site-packages\wikipedia\wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("html.parser"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file c:\Users\harsh\AppData\Local\Programs\Python\Python39\lib\site-packages\wikipedia\wikipedia.py. To get rid of this warning, pass the additional argument 'features="html.parser"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


In [6]:
llm = OllamaLLM(model="llama3.2:1b")
llm_transformer = LLMGraphTransformer(llm=llm)

# Extract graph data
graph_documents = llm_transformer.convert_to_graph_documents(docs)

In [7]:
# Store to neo4j
graph.add_graph_documents(
  graph_documents, 
  baseEntityLabel=True, 
  include_source=True)

In [8]:
from sentence_transformers import SentenceTransformer
from langchain.vectorstores import Neo4jVector
from langchain_core.embeddings import Embeddings  # Updated import

model = SentenceTransformer('all-MiniLM-L6-v2')

In [9]:
from typing import List

class Sent(Embeddings):
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return model.encode(texts).tolist()
    
    def embed_query(self, text: str) -> List[float]:
        return model.encode([text]).tolist()[0]

se = Sent()

In [10]:
# Now, create the vector index using the custom embeddings
vector_index = Neo4jVector.from_existing_graph(
    se,
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding"
)

In [11]:
from typing import List
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

# Simple Entities Model
class Entities(BaseModel):
    names: List[str] = Field(default_factory=list)

# Simple Prompt
prompt = ChatPromptTemplate.from_messages([
    (
        "system", 
        "Extract relevant entities from the text about Empty Nest Syndrome. Focus on identifying:"
        " 1. People (e.g., psychologists, researchers, authors who have written about the topic)"
        " 2. Organizations (e.g., mental health associations, support groups)"
        " 3. Psychological or medical terms"
        " 4. Related life stages or family dynamics concepts"
    ),
    (
        "human", 
        "Text: {question}\n\nList key entities:"
    )
])

# Entity Extraction Chain
entity_chain = prompt | llm


def extract_entities(question):
    response = entity_chain.invoke({"question": question})
    
    try:
        # Check if response is already a string or has a content attribute
        if hasattr(response, 'content'):
            text_to_parse = response.content
        else:
            text_to_parse = str(response)
        
        # More robust parsing
        entity_names = []
        
        # Specific parsing for the structured response
        sections = {
            'People': [],
            'Organizations': [],
            'Psychological Terms': [],
            'Life Stages': []
        }
        
        # Parsing logic
        current_section = None
        for line in text_to_parse.split('\n'):
            line = line.strip()
            
            # Detect section headers
            if line.startswith('1. People:'):
                current_section = 'People'
                continue
            elif line.startswith('2. Organizations:'):
                current_section = 'Organizations'
                continue
            elif line.startswith('3. Psychological'):
                current_section = 'Psychological Terms'
                continue
            elif line.startswith('4. Related'):
                current_section = 'Life Stages'
                continue
            
            # Parse entities within sections
            if current_section and line.startswith('-'):
                # Extract entity, removing parenthetical descriptions
                entity = line.split('-')[1].split('(')[0].strip()
                if entity:
                    sections[current_section].append(entity)
                    entity_names.append(entity)
        
        # Fallback if no entities found
        if not entity_names:
            entity_names = [
                name.strip() 
                for name in text_to_parse.split(',') 
                if name.strip()
            ]
        
        # Print out parsed sections for debugging
        print("Parsed Entities:")
        for section, entities in sections.items():
            print(f"{section}: {entities}")
        
        return Entities(names=entity_names)
    
    except Exception as e:
        print(f"Error parsing entities: {e}")
        # Fallback to simple parsing
        return Entities(names=[
            name.strip() 
            for name in str(response).split(',') 
            if name.strip()
        ])


# Create Neo4j Full-Text Index for Entities
graph.query(
    "CREATE FULLTEXT INDEX entity IF NOT EXISTS FOR (e:__Entity__) ON EACH [e.id]"
)


[]

In [12]:
# Method 1: Direct parsing
result = entity_chain.invoke({"question": "What is empty nest syndrome?"})
print("Extracted Entities:", result)

# Method 2: Manual parsing
response = entity_chain.invoke({"question": "What is empty nest syndrome?"})
entities = [name.strip() for name in response.split(',') if name.strip()]
print("Extracted Entities:", entities)

# Full response
full_response = llm.invoke("What is empty nest syndrome?")
print("\nFull Explanation:")
print(full_response)

Extracted Entities: Based on the text, here are the relevant entities extracted:

1. People:
   - Dr. Robert Berkman (psychologist who has written about Empty Nest Syndrome)
   - David Barker (researcher who has studied Empty Nest Syndrome)

2. Organizations:
   - National Alliance on Mental Illness (NAMI) - organization that supports individuals with mental health conditions, including those experiencing Empty Nest Syndrome
   - American Psychological Association (APA) - organization that has published research on Empty Nest Syndrome

3. Psychological or medical terms:
   - Empty nest syndrome - condition characterized by feelings of loneliness and isolation after the end of a child-rearing period
   - Postnatal depression - condition where women experience emotional distress after giving birth to their children
   - Social isolation - feeling disconnected from others, often resulting in loneliness

4. Related life stages or family dynamics concepts:
   - Childhood - stage of life whe

In [13]:
def generate_full_text_query(input: str) -> str:
    """
    Generate a simplified full-text search query
    """
    # Remove any special characters and split into words
    words = input.replace(',', ' ').replace('-', ' ').split()
    
    # Join words with OR for more flexible matching
    return ' OR '.join([f"{word}~2" for word in words])


def structured_retriever(question: str) -> str:
    """
    Collects the neighborhood of entities mentioned in the question
    """
    result = ""
    
    # Extract entities using the updated function
    entities_response = extract_entities(question)
    
    # Use the names directly from the Entities object
    entities = entities_response.names
    
    # Iterate through entities
    for entity in entities:
        try:
            response = graph.query(
                """CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})
                YIELD node,score
                CALL {
                  WITH node
                  MATCH (node)-[r:!MENTIONS]->(neighbor)
                  RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
                  UNION ALL
                  WITH node
                  MATCH (node)<-[r:!MENTIONS]-(neighbor)
                  RETURN neighbor.id + ' - ' + type(r) + ' -> ' +  node.id AS output
                }
                RETURN output LIMIT 50
                """,
                {"query": generate_full_text_query(entity)},
            )
            result += "\n".join([el['output'] for el in response])
        except Exception as e:
            print(f"Error processing entity {entity}: {e}")
    
    return result





In [14]:
# Try running the retriever
print(structured_retriever("What is Empty Nest Syndrome?"))

Parsed Entities:
People: []
Organizations: []
Psychological Terms: []
Life Stages: []
Error processing entity here are the relevant entities extracted:

1. People:
	* Dr. Gary E. Schwartz (psychologist who wrote about Empty Nest Syndrome)
2. Organizations:
	* National Alliance for the Mentally Ill (NAMI) - a mental health association
	* The Empty Nest Society (organization founded by Dr. Gary E. Schwartz to raise awareness and support for empty nesters)
3. Psychological or medical terms:
	* Empty Nest Syndrome (a psychological condition characterized by feelings of sadness: {code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `db.index.fulltext.queryNodes`: Caused by: org.apache.lucene.queryparser.classic.ParseException: Encountered " <FUZZY_SLOP> "~2 "" at line 1, column 66.
Was expecting one of:
    <BAREOPER> ...
    "(" ...
    "*" ...
    <QUOTED> ...
    <TERM> ...
    <PREFIXTERM> ...
    <WILDTERM> ...
    <REGEXPTERM> ...
    "[" ...
    "

In [15]:
def retriever(question: str):
    print(f"Search query: {question}")
    structured_data = structured_retriever(question)
    unstructured_data = [el.page_content for el in vector_index.similarity_search(question)]
    final_data = f"""Structured data:
{structured_data}
Unstructured data:
{"#Document ". join(unstructured_data)}
    """
    return final_data

In [24]:
_template = """Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question,
in its original language.
Chat History:
{chat_history}
Follow Up Input: {question}
Standalone question:"""

CONDENSE_QUESTION_PROMPT = PromptTemplate.from_template(_template)

def _format_chat_history(chat_history: List[Tuple[str, str]]) -> List:
    buffer = []
    for human, ai in chat_history:
        buffer.append(HumanMessage(content=human))
        buffer.append(AIMessage(content=ai))
    return buffer

_search_query = RunnableBranch(
    # If input includes chat_history, we condense it with the follow-up question
    (
        RunnableLambda(lambda x: bool(x.get("chat_history"))).with_config(
            run_name="HasChatHistoryCheck"
        ),
        RunnablePassthrough.assign(
            chat_history=lambda x: _format_chat_history(x["chat_history"])
        )
        | CONDENSE_QUESTION_PROMPT
        | OllamaLLM(model="llama3.2:1b", temperature=0.7)  # Replaced ChatOpenAI with OllamaLLM
        | StrOutputParser(),
    ),
    # Else, we have no chat history, so just pass through the question
    RunnableLambda(lambda x : x["question"]),
)

In [ ]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
Use natural language and be concise.
Answer:"""

prompt = ChatPromptTemplate.from_template(template)

chain = (
    RunnableParallel(
        {
            "context": _search_query | retriever,  # Corrected from query
            "question": RunnablePassthrough(),
        }
    )
    | prompt
    | OllamaLLM(model="llama3.2:1b")  # Replaced 'llm' with OllamaLLM
    | StrOutputParser()
)

In [26]:
chain.invoke({"question": "What are symptoms around Empty Nest Syndrome?"})

'Empty Nest Syndrome is often marked by feelings of anxiety, sadness, or restlessness. Symptoms can include:\n\n* Unexplained changes in mood\n* Withdrawal from social activities\n* Difficulty adjusting to new routines\n* Increased sleepiness or insomnia\n* Changes in appetite or interest in sex\n* Feeling disconnected from family members'

In [27]:
chain.invoke(
    {
        "question": "Who experiences Empty Nest Syndrome?",
        "chat_history": [("What are symptoms around Empty Nest Syndrome?", "Empty Nest Syndrome is often marked by feelings of anxiety, sadness, or restlessness. Symptoms can include:\n\n* Unexplained changes in mood\n* Withdrawal from social activities\n* Difficulty adjusting to new routines\n* Increased sleepiness or insomnia\n* Changes in appetite or interest in sex\n* Feeling disconnected from family members")],
    }
)

"Empty Nest Syndrome can affect anyone, but it's often associated with parents who have recently lost their child. This transition can lead to feelings of anxiety, sadness, or restlessness as they adjust to new life without the daily care and attention of a child."